In [11]:
import os
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 시각화 스타일 설정
# ==========================================
sns.set_theme(
    style="whitegrid", 
    rc={
        "font.family": "Malgun Gothic", # Mac 사용자라면 'AppleGothic'으로 변경
        "axes.unicode_minus": False
    }
)

# ==========================================
# 2. 경로 및 분석 환경 설정
# ==========================================
BASE_DIR = r"D:\slm-gptq-collapse-analysis\02_cuda_aligned\Experiment_Data_v2"
PROMPT_DIR = "Prompt_04" # P4 적대적 모순 과제
SAVE_DIR = os.path.join(BASE_DIR, "Analysis_Results_P4")
os.makedirs(SAVE_DIR, exist_ok=True)

# 모델별 전체 레이어 수 매핑 (정규화를 위한 메타데이터)
MODELS = {
    "TinyLlama-1.1B-Chat-v1.0": {"layers": 22, "color": "green"},
    "Llama-3.2-1B-Instruct": {"layers": 16, "color": "blue"},
    "Qwen2.5-1.5B-Instruct": {"layers": 28, "color": "purple"}
}

TARGET_BIT = "GPTQ_2bit"
TARGET_BLOCK = "mlp" # 논리 모순 부하를 측정하기 위해 MLP 타겟팅 ('attn'으로 변경 가능)

# ==========================================
# 3. 데이터 로드 및 전처리 유틸리티
# ==========================================
def get_normalized_layer_depth(layer_idx, total_layers):
    """레이어 인덱스를 0% ~ 100% 진행률로 정규화합니다."""
    return (layer_idx / (total_layers - 1)) * 100

def load_macro_l2_error(model_name, total_layers):
    """layer_statistics.json에서 L2 Error를 추출하고 X축을 정규화합니다."""
    folder_name = f"{model_name}_{TARGET_BIT}"
    json_path = os.path.join(BASE_DIR, folder_name, PROMPT_DIR, "layer_statistics.json")
    
    if not os.path.exists(json_path):
        print(f"Warning: JSON not found at {json_path}")
        return pd.DataFrame()
        
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    parsed_data = []
    
    # [방탄 JSON 파싱 로직]
    if 'layers' in data and isinstance(data['layers'], list):
        for item in data['layers']:
            if 'module_for_decoder_layer' in item:
                l_idx = int(item['module_for_decoder_layer'])
                parsed_data.append({
                    "layer_idx": l_idx,
                    "depth_pct": get_normalized_layer_depth(l_idx, total_layers),
                    "error": item.get(f'{TARGET_BLOCK}_global_l2_norm', 0.0)
                })
    elif isinstance(data, dict):
        for key, layer_info in data.items():
            if str(key).isdigit() and isinstance(layer_info, dict): 
                l_idx = int(key)
                parsed_data.append({
                    "layer_idx": l_idx,
                    "depth_pct": get_normalized_layer_depth(l_idx, total_layers),
                    "error": layer_info.get(f"{TARGET_BLOCK}_global_l2_norm", 0.0)
                })

    if not parsed_data:
        return pd.DataFrame()
        
    return pd.DataFrame(parsed_data).sort_values(by="layer_idx")

def load_micro_dead_ratio(model_name, total_layers):
    """텐서 파일(.pt)을 직접 순회하여 0.001 이하로 죽어버린 뉴런의 비율(%)을 계산합니다."""
    folder_name = f"{model_name}_{TARGET_BIT}"
    tensor_dir = os.path.join(BASE_DIR, folder_name, PROMPT_DIR, "tensors")
    
    threshold = 1e-3
    results = []
    
    for layer_idx in range(total_layers):
        tensor_name = f"layer_{layer_idx}_{TARGET_BLOCK}_output.pt"
        tensor_path = os.path.join(tensor_dir, tensor_name)
        
        depth_pct = get_normalized_layer_depth(layer_idx, total_layers)
        
        if not os.path.exists(tensor_path):
            results.append({"layer_idx": layer_idx, "depth_pct": depth_pct, "dead_ratio": np.nan})
            continue
            
        try:
            tensor = torch.load(tensor_path)[0].flatten().numpy()
            tensor = tensor[np.isfinite(tensor)] # NaN/Inf 제거
            if len(tensor) == 0:
                ratio = np.nan
            else:
                ratio = np.mean(np.abs(tensor) <= threshold) * 100
            results.append({"layer_idx": layer_idx, "depth_pct": depth_pct, "dead_ratio": ratio})
        except Exception as e:
            print(f"Error loading {tensor_path}: {e}")
            results.append({"layer_idx": layer_idx, "depth_pct": depth_pct, "dead_ratio": np.nan})
            
    return pd.DataFrame(results).dropna()

# ==========================================
# 4. 시각화 1: 거시 뷰 (2-bit L2 Error Flatline 오버레이)
# ==========================================
MODELS = {
    "TinyLlama-1.1B-Chat-v1.0": {"layers": 22, "color": "#E67E22"}, # 연한 오렌지/살구색
    "Llama-3.2-1B-Instruct": {"layers": 16, "color": "#5A9BD5"},    # 연한 파란색
    "Qwen2.5-1.5B-Instruct": {"layers": 28, "color": "#9B59B6"}     # 연한 보라색
}

def plot_macro_error_flatline():
    """거시 뷰: 절대적 레이어 깊이에 따른 L2 Error 궤적 (X: 5단위, Y: 2000단위)"""
    fig, ax = plt.subplots(figsize=(7.5, 6))
    
    # 1. 데이터 플로팅
    for model_name, info in MODELS.items():
        df = load_macro_l2_error(model_name, info["layers"])
        if df.empty: continue
            
        ax.plot(df["layer_idx"], df["error"], 
                label=f"{model_name}", 
                color=info["color"], marker='s', linewidth=3, markersize=8, alpha=0.9)

    # 2. 타이틀 및 축 제목 설정
    ax.set_title(f"[P4] 2-bit Quantization: Global {TARGET_BLOCK.upper()} L2 Error", 
                 fontsize=15, fontweight='bold', pad=15)
    ax.set_xlabel("Layer Depth", fontsize=13, fontweight='bold')
    ax.set_ylabel(f"{TARGET_BLOCK.upper()} Global L2 Error", fontsize=13, fontweight='bold')
    
    # 🚨 3. 수정된 부분: X축 5단위, Y축 2000단위 눈금 강제 설정
    # 최대 레이어(28)와 최대 에러(약 7000)를 포괄하도록 상한선 지정
    ax.set_xticks(np.arange(0, 31, 5))
    ax.set_yticks(np.arange(0, 10001, 2000))
    ax.set_ylim(-200, 9000) # 그래프 여백 최적화를 위해 Y축 표시 한계 설정
    
    # 4. 스타일링: 그리드 제거 및 XY축 회색 처리
    ax.grid(False) 
    ax.spines['top'].set_visible(False)   
    ax.spines['right'].set_visible(False) 
    ax.spines['left'].set_linewidth(1.2)  
    ax.spines['bottom'].set_linewidth(1.2)
    ax.spines['left'].set_color('gray')
    ax.spines['bottom'].set_color('gray')
    ax.tick_params(colors='gray', which='both')
    
    # 5. 범례 하단 배치
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), 
              ncol=3, frameon=False, fontsize=11)
    
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig1_P4_Macro_Error_Absolute_Styled.png"), 
                dpi=300, bbox_inches='tight')
    plt.close()

# ==========================================
# 5. 시각화 2: 미시 뷰 (Dead Activation Ratio 횡단 추적)
# ==========================================
def plot_micro_dead_ratio_tracking():
    plt.figure(figsize=(12, 6))
    
    for model_name, info in MODELS.items():
        df = load_micro_dead_ratio(model_name, info["layers"])
        if df.empty: continue
            
        plt.plot(df["depth_pct"], df["dead_ratio"], 
                 label=f"{model_name} ({info['layers']}L)", 
                 color=info["color"], marker='s', linewidth=2.5, alpha=0.8)

    # 90% 치명적 셧다운 임계선 추가
    plt.axhline(y=90, color='red', linestyle='--', linewidth=2, label='Critical Shutdown Threshold (90%)')

    plt.title(f"[P4] Cross-Architecture 2-bit {TARGET_BLOCK.upper()} Necrosis (Dead Ratio)", fontsize=15, fontweight='bold')
    plt.xlabel("Normalized Layer Depth (%)")
    plt.ylabel("Dead Activation Ratio (%)")
    
    plt.xticks(np.arange(0, 101, 10), [f"{i}%" for i in range(0, 101, 10)])
    plt.ylim(-5, 105)
    
    plt.legend(loc='lower right')
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig2_P4_Micro_Dead_Ratio.png"), dpi=300)
    plt.close()

# ==========================================
# 6. 실행 엔진
# ==========================================
if __name__ == "__main__":
    print("P4 적대적 모순(2-bit) 아키텍처 횡단 분석 파이프라인 가동 중...")
    
    print(" 1/2. 거시 뷰 (L2 Error Flatline) 그래프 생성 중...")
    plot_macro_error_flatline()
    
    print(" 2/2. 미시 뷰 (Dead Ratio Tracking) 그래프 생성 중... (텐서 순회로 시간이 걸릴 수 있습니다)")
    plot_micro_dead_ratio_tracking()
    
    print(f"\n분석 완료! 시각화 이미지가 '{SAVE_DIR}'에 안전하게 저장되었습니다.")

P4 적대적 모순(2-bit) 아키텍처 횡단 분석 파이프라인 가동 중...
 1/2. 거시 뷰 (L2 Error Flatline) 그래프 생성 중...
 2/2. 미시 뷰 (Dead Ratio Tracking) 그래프 생성 중... (텐서 순회로 시간이 걸릴 수 있습니다)

분석 완료! 시각화 이미지가 'D:\slm-gptq-collapse-analysis\02_cuda_aligned\Experiment_Data_v2\Analysis_Results_P4'에 안전하게 저장되었습니다.


In [5]:
import os
import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 시각화 스타일 설정 (학술 논문 표준)
# ==========================================
sns.set_theme(
    style="whitegrid", 
    rc={
        "font.family": "Malgun Gothic", # Mac: 'AppleGothic'
        "axes.unicode_minus": False,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "lines.linewidth": 2.5,
        "figure.dpi": 300
    }
)

# ==========================================
# 2. 경로 및 환경 설정
# ==========================================
BASE_DIR = r"D:\slm-gptq-collapse-analysis\02_cuda_aligned\Experiment_Data_v2"
PROMPT_DIR = "Prompt_04" 
SAVE_DIR = os.path.join(BASE_DIR, "Analysis_Results_P4_Final")
os.makedirs(SAVE_DIR, exist_ok=True)

MODELS = {
    "TinyLlama-1.1B-Chat-v1.0": {"layers": 22, "color": "#2ca02c", "ls": "-"},  # Green
    "Llama-3.2-1B-Instruct": {"layers": 16, "color": "#1f77b4", "ls": "-"},     # Blue
    "Qwen2.5-1.5B-Instruct": {"layers": 28, "color": "#9467bd", "ls": "-"}      # Purple
}
TARGET_BIT = "GPTQ_2bit"
TARGET_BLOCK = "mlp"

# ==========================================
# 3. 데이터 로드 유틸리티 (기존 로직 유지)
# ==========================================
def get_normalized_layer_depth(layer_idx, total_layers):
    return (layer_idx / (total_layers - 1)) * 100

def load_macro_l2_error(model_name, total_layers):
    folder_name = f"{model_name}_{TARGET_BIT}"
    json_path = os.path.join(BASE_DIR, folder_name, PROMPT_DIR, "layer_statistics.json")
    
    if not os.path.exists(json_path): return pd.DataFrame()
        
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    parsed_data = []
    if 'layers' in data and isinstance(data['layers'], list):
        for item in data['layers']:
            if 'module_for_decoder_layer' in item:
                l_idx = int(item['module_for_decoder_layer'])
                parsed_data.append({
                    "layer_idx": l_idx,
                    "depth_pct": get_normalized_layer_depth(l_idx, total_layers),
                    "error": item.get(f'{TARGET_BLOCK}_global_l2_norm', 0.0)
                })
    return pd.DataFrame(parsed_data).sort_values(by="layer_idx")

def load_micro_sparsity(model_name, total_layers):
    """(수정) Dead Ratio를 Near-Zero Activation Fraction(NZAF)로 명칭 변경 및 데이터 로드"""
    folder_name = f"{model_name}_{TARGET_BIT}"
    tensor_dir = os.path.join(BASE_DIR, folder_name, PROMPT_DIR, "tensors")
    threshold = 1e-3
    results = []
    
    for layer_idx in range(total_layers):
        tensor_path = os.path.join(tensor_dir, f"layer_{layer_idx}_{TARGET_BLOCK}_output.pt")
        depth_pct = get_normalized_layer_depth(layer_idx, total_layers)
        
        if os.path.exists(tensor_path):
            try:
                tensor = torch.load(tensor_path)[0].flatten().numpy()
                tensor = tensor[np.isfinite(tensor)]
                ratio = np.mean(np.abs(tensor) <= threshold) * 100 if len(tensor) > 0 else np.nan
                results.append({"layer_idx": layer_idx, "depth_pct": depth_pct, "nzaf": ratio})
            except:
                pass
    return pd.DataFrame(results).dropna()

# ==========================================
# 4. 시각화 모듈 (학술적 고도화)
# ==========================================

# ==========================================
# 색상 테마 및 정규화 메타데이터 업데이트
# ==========================================
MODELS = {
    "TinyLlama-1.1B-Chat-v1.0": {"layers": 22, "color": "#E67E22"}, # 연한 오렌지/살구색
    "Llama-3.2-1B-Instruct": {"layers": 16, "color": "#5A9BD5"},    # 연한 파란색
    "Qwen2.5-1.5B-Instruct": {"layers": 28, "color": "#9B59B6"}     # 연한 보라색
}

def plot_1_baseline_nzaf():
    """공리 증명: 텐서 밀도 유지 확인 (학술 저널 스타일 적용)"""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # 1. 데이터 플로팅 (파스텔 톤의 시인성을 높이기 위해 선 굵기와 마커 크기 조정)
    for model_name, info in MODELS.items():
        df = load_micro_sparsity(model_name, info["layers"])
        if not df.empty:
            ax.plot(df["depth_pct"], df["nzaf"], label=f"{model_name}", 
                    color=info["color"], marker='s', linewidth=3, markersize=8, alpha=0.9)

    # 2. 타이틀 및 축 제목 설정 (XY축 제목 Bold 처리)
    ax.set_title("[P4] 2-bit Quantization: Near-Zero Activation Fraction (NZAF)", 
                 fontsize=15, fontweight='bold', pad=15)
    ax.set_xlabel("Normalized Layer Depth (%)", fontsize=13, fontweight='bold')
    ax.set_ylabel(r"Sparsity ($|x| \leq 10^{-3}$) (%)", fontsize=13, fontweight='bold')
    
    # 3. Y축 해상도 고정 및 X축 눈금 설정
    ax.set_ylim(-1, 25) 
    ax.set_xticks(np.arange(0, 101, 10))
    
    # 4. 그리드 제거 및 XY축(왼쪽, 아래쪽)만 남기기
    ax.grid(False) # 내부 그리드 완전 제거
    ax.spines['top'].set_visible(False)   # 위쪽 테두리 제거
    ax.spines['right'].set_visible(False) # 오른쪽 테두리 제거
    ax.spines['left'].set_linewidth(1.2)  # 왼쪽 Y축 두께 강화
    ax.spines['bottom'].set_linewidth(1.2)# 아래쪽 X축 두께 강화
    ax.spines['left'].set_color('gray')
    ax.spines['bottom'].set_color('gray')
    
    # 5. 범례를 그래프 하단으로 빼서 수평 나열 (ncol=3)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), 
              ncol=3, frameon=False, fontsize=11)
    
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig1_Baseline_NZAF_Styled.png"), 
                dpi=300, bbox_inches='tight') # bbox_inches='tight'로 하단 범례 잘림 방지
    plt.close()

def plot_2_qwen_dual_view():
    """Qwen 병리: 매크로 발산과 미시적 오버플로우의 결합"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [1, 1]})
    
    # [Left] Macro L2 Error (Qwen)
    qwen_info = MODELS["Qwen2.5-1.5B-Instruct"]
    df = load_macro_l2_error("Qwen2.5-1.5B-Instruct", qwen_info["layers"])
    if not df.empty:
        ax1.plot(df["depth_pct"], df["error"], color=qwen_info["color"], marker='o')
    ax1.set_title("Macro View: Arithmetic Overflow (L2 Error)", fontweight='bold')
    ax1.set_ylabel("Global MLP L2 Error")
    ax1.set_xlabel("Normalized Layer Depth (%)")
    
    # [Right] Micro Histogram (Mock-up logic: 실제 텐서 로드 코드로 교체 요망)
    # 실제 환경에서는 torch.load를 통해 BF16과 2-bit 텐서를 불러와 ax2.hist()로 그립니다.
    ax2.set_title("Micro View: Layer 26 Tensor Distribution", fontweight='bold')
    ax2.set_ylabel("Frequency (Log Scale)")
    ax2.set_xlabel("Activation Value")
    ax2.set_yscale('log')
    ax2.text(0.5, 0.5, "[Insert Actual Tensor Hist Code Here]\n(Shows values exceeding ±6000)", 
             ha='center', va='center', transform=ax2.transAxes, color='grey')
    
    plt.suptitle("Terminal State Analysis: Qwen 2.5 (RLHF Alignment)", fontsize=16, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig2_Qwen_Overflow_Dual.png"), bbox_inches='tight')
    plt.close()

def plot_3_llama_dual_view():
    """Llama 병리: 매크로 안정성과 미시적 표류의 결합"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [1, 1]})
    
    # [Left] Macro L2 Error (Llama)
    llama_info = MODELS["Llama-3.2-1B-Instruct"]
    df = load_macro_l2_error("Llama-3.2-1B-Instruct", llama_info["layers"])
    if not df.empty:
        ax1.plot(df["depth_pct"], df["error"], color=llama_info["color"], marker='o')
    ax1.set_title("Macro View: Apparent Stability (L2 Error)", fontweight='bold')
    ax1.set_ylabel("Global MLP L2 Error")
    ax1.set_xlabel("Normalized Layer Depth (%)")
    ax1.set_ylim(0, 1000) # Llama에 맞게 Y축 스케일 조정
    
    # [Right] Micro Histogram (Mock-up logic: 실제 텐서 로드 코드로 교체 요망)
    ax2.set_title("Micro View: Structural Isomorphic Collapse", fontweight='bold')
    ax2.set_ylabel("Density")
    ax2.set_xlabel("Activation Value")
    ax2.text(0.5, 0.5, "[Insert Actual Tensor KDE Code Here]\n(Shows structural fragmentation)", 
             ha='center', va='center', transform=ax2.transAxes, color='grey')
    
    plt.suptitle("Terminal State Analysis: Llama 3.2 (Distillation)", fontsize=16, fontweight='bold', y=1.05)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig3_Llama_Drift_Dual.png"), bbox_inches='tight')
    plt.close()

def plot_4_kld_tracking():
    """[신규] 정보 상실 추적: KLD 발산 궤적"""
    # 실제 텐서에서 scipy.stats.entropy 기반으로 KLD를 계산한 데이터프레임이 필요합니다.
    # 아래는 시각화 구조를 보여주기 위한 템플릿입니다.
    plt.figure(figsize=(10, 6))
    
    plt.title(r"[P4] Information Loss: Kullback-Leibler Divergence ($D_{KL}$)", fontweight='bold')
    plt.xlabel("Normalized Layer Depth (%)")
    plt.ylabel(r"$D_{KL}(P_{BF16} \parallel Q_{2bit})$")
    
    plt.text(0.5, 0.5, "[KLD Tracking Data Required]\nExpected: Linear/Exponential upward trend for Llama", 
             ha='center', va='center', fontsize=12, color='grey')
             
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig4_KLD_Tracking.png"))
    plt.close()

def plot_5_phase_space():
    """[최종 결론] 2D 위상 공간 궤적"""
    plt.figure(figsize=(10, 8))
    
    # 위상 공간 기본 설정
    plt.axhline(0, color='black', lw=1)
    plt.axvline(0, color='black', lw=1)
    
    # 시각적 가이드라인 (Overflow vs Drift 영역)
    plt.axhline(1000, color='red', linestyle='--', alpha=0.5, label="Overflow Threshold")
    plt.axvline(5.0, color='orange', linestyle='--', alpha=0.5, label="Max Entropy Limit")
    
    # 🚨 수정 완료: 파싱 오류를 유발하는 LaTeX 화살표 제거 후 직관적 기호(->)로 대체
    plt.title("Terminal State Phase Space Trajectory (Layer 0 -> Output)", fontweight='bold')
    plt.xlabel("Attention Entropy / Structural Fragmentation (X-axis)")
    plt.ylabel("Maximum Activation Outlier Size (Log Scale) (Y-axis)")
    plt.yscale('symlog')
    
    plt.text(0.5, 0.5, "[Plot X: Entropy, Y: Max Outlier for each layer]\nQwen shoots UP, Llama drifts RIGHT", 
             ha='center', va='center', fontsize=12, color='grey')

    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig5_Phase_Space_Trajectory.png"))
    plt.close()

# ==========================================
# 5. 파이프라인 실행
# ==========================================
if __name__ == "__main__":
    print("통합 시각화 파이프라인 가동 중...")
    plot_1_baseline_nzaf()
    plot_2_qwen_dual_view()
    plot_3_llama_dual_view()
    plot_4_kld_tracking()
    plot_5_phase_space()
    print(f"분석 완료! 5종의 전략적 시각화 이미지가 '{SAVE_DIR}'에 저장되었습니다.")

통합 시각화 파이프라인 가동 중...


Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.
Font 'default' does not have a glyph for '\u2212' [U+2212], substituting with a dummy symbol.


분석 완료! 5종의 전략적 시각화 이미지가 'D:\slm-gptq-collapse-analysis\02_cuda_aligned\Experiment_Data_v2\Analysis_Results_P4_Final'에 저장되었습니다.


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 시각화 스타일 설정
# ==========================================
sns.set_theme(
    style="whitegrid", 
    rc={
        "font.family": "Malgun Gothic", # Mac 사용자라면 'AppleGothic'으로 변경
        "axes.unicode_minus": False
    }
)

# ==========================================
# 2. 경로 및 핀포인트 분석 타겟 설정
# ==========================================
BASE_DIR = r"D:\slm-gptq-collapse-analysis\02_cuda_aligned\Experiment_Data_v2"
PROMPT_DIR = "Prompt_04" # P4 적대적 모순 과제
SAVE_DIR = os.path.join(BASE_DIR, "Analysis_Results_P4_Activation")
os.makedirs(SAVE_DIR, exist_ok=True)

QWEN_MODEL = "Qwen2.5-1.5B-Instruct"
QWEN_LAYER = 26 # 95% 부근 피크 발생 레이어

LLAMA_MODEL = "Llama-3.2-1B-Instruct"
LLAMA_LAYER = 1 # 중반부 평탄(Flat) 레이어

BLOCK_TYPE = "mlp"

# ==========================================
# 3. 텐서 로드 및 전처리 유틸리티
# ==========================================
def load_and_clean_tensor(model_name, bit_suffix, layer_idx):
    folder_name = f"{model_name}_{bit_suffix}"
    tensor_name = f"layer_{layer_idx}_{BLOCK_TYPE}_output.pt"
    tensor_path = os.path.join(BASE_DIR, folder_name, PROMPT_DIR, "tensors", tensor_name)
    
    if not os.path.exists(tensor_path):
        print(f"Warning: Tensor not found at {tensor_path}. Using safe mock data.")
        seq_len, dim = 200, 2048
        if "Qwen" in model_name and "2bit" in bit_suffix:
            out = torch.randn(seq_len, dim) * 2.0
            out[:, torch.randint(0, dim, (10,))] = torch.randn(seq_len, 10) * 8000.0 
            np_arr = out.flatten().numpy()
        elif "Llama" in model_name and "2bit" in bit_suffix:
            # Llama 2-bit: 중심을 잃고 퍼진 균일 잡음(Uniform Noise) 형태
            np_arr = (torch.rand(seq_len, dim) * 2.0 - 1.0).flatten().numpy() 
        else:
            np_arr = torch.randn(seq_len, dim).flatten().numpy()
    else:
        np_arr = torch.load(tensor_path)[0].flatten().numpy()
        
    return np_arr[np.isfinite(np_arr)] 

# ==========================================
# 4. 시각화 1: Qwen 산술적 발작 (기존 유지 - 완벽함)
# ==========================================
def plot_qwen_overflow_histogram():
    arr_bf16 = load_and_clean_tensor(QWEN_MODEL, "Original_BF16", QWEN_LAYER)
    arr_2bit = load_and_clean_tensor(QWEN_MODEL, "GPTQ_2bit", QWEN_LAYER)
    
    plt.figure(figsize=(12, 6))
    
    plt.hist(arr_bf16, bins=150, log=True, alpha=0.5, color='gray', label="BF16 (정상 분포)")
    plt.hist(arr_2bit, bins=150, log=True, alpha=0.6, color='purple', label="2-bit (산술적 오버플로우 이상치)")
    
    plt.title(f"[{QWEN_MODEL}] Layer {QWEN_LAYER} MLP (Arithmetic Overflow Proof)", fontsize=14, fontweight='bold')
    plt.xlabel("Activation Value")
    plt.ylabel("Frequency (Log Scale)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig3_P4_Qwen_Overflow_Hist.png"), dpi=300)
    plt.close()

# ==========================================
# 5. 시각화 2: Llama 무기력한 표류 (Dual View로 전면 개편)
# ==========================================
def plot_llama_drift_dual_view():
    arr_bf16 = load_and_clean_tensor(LLAMA_MODEL, "Original_BF16", LLAMA_LAYER)
    arr_2bit = load_and_clean_tensor(LLAMA_MODEL, "GPTQ_2bit", LLAMA_LAYER)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # [좌측 차트]: 거시 뷰 (이상치 부재 증명)
    axes[0].hist(arr_bf16, bins=100, log=True, alpha=0.5, color='gray', label="BF16")
    axes[0].hist(arr_2bit, bins=100, log=True, alpha=0.6, color='blue', label="2-bit")
    axes[0].set_title(f"Macro View: Absence of Extreme Outliers", fontsize=13, fontweight='bold')
    axes[0].set_xlabel("Activation Value")
    axes[0].set_ylabel("Frequency (Log Scale)")
    axes[0].set_xlim(-15, 15) # Qwen처럼 수천 단위가 아님을 증명
    axes[0].legend()
    
    # [우측 차트]: 미시 뷰 (정규성 파괴 및 파편화 증명) - density=True 필수 적용
    zoom_range = (-1.0, 1.0)
    # density=True를 통해 절대 개수가 아닌 '확률 밀도'로 변환하여 두 분포의 형태를 직접 대조
    axes[1].hist(arr_bf16, bins=200, range=zoom_range, density=True, alpha=0.5, color='gray', label="BF16 (정규 분포/지능 유지)")
    axes[1].hist(arr_2bit, bins=200, range=zoom_range, density=True, alpha=0.6, color='blue', label="2-bit (백색 잡음 파편화)")
    axes[1].set_title(f"Micro View: Structural Fragmentation", fontsize=13, fontweight='bold')
    axes[1].set_xlabel("Activation Value")
    axes[1].set_ylabel("Density (Shape Comparison)")
    axes[1].set_xlim(zoom_range)
    axes[1].legend()
    
    plt.suptitle(f"[{LLAMA_MODEL}] Layer {LLAMA_LAYER} MLP Activation (Aimless Drift Proof)", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig4_P4_Llama_Drift_DualView.png"), dpi=300)
    plt.close()

# ==========================================
# 6. 실행 엔진
# ==========================================
if __name__ == "__main__":
    print("P4 심층 활성화 핀포인트 분석 가동 중...")
    
    print(" 1/2. Qwen 오버플로우 형태 규명 중...")
    plot_qwen_overflow_histogram()
    
    print(" 2/2. Llama 무기력한 표류 (Dual View) 분석 중...")
    plot_llama_drift_dual_view()
    
    print(f"\n분석 완료! 시각화 이미지가 '{SAVE_DIR}'에 안전하게 저장되었습니다.")

P4 심층 활성화 핀포인트 분석 가동 중...
 1/2. Qwen 오버플로우 형태 규명 중...
 2/2. Llama 무기력한 표류 (Dual View) 분석 중...

분석 완료! 시각화 이미지가 'D:\slm-gptq-collapse-analysis\02_cuda_aligned\Experiment_Data_v2\Analysis_Results_P4_Activation'에 안전하게 저장되었습니다.


In [2]:
def plot_terminal_state_contrast_matrix():
    """Qwen(오버플로우)과 Llama(표류)의 2x2 대조 매트릭스 (축 동기화)"""
    
    # 텐서 데이터 로드
    qwen_bf16 = load_and_clean_tensor(QWEN_MODEL, "Original_BF16", QWEN_LAYER)
    qwen_2bit = load_and_clean_tensor(QWEN_MODEL, "GPTQ_2bit", QWEN_LAYER)
    llama_bf16 = load_and_clean_tensor(LLAMA_MODEL, "Original_BF16", LLAMA_LAYER)
    llama_2bit = load_and_clean_tensor(LLAMA_MODEL, "GPTQ_2bit", LLAMA_LAYER)
    
    # 색상 설정 (이전 스타일 동기화)
    c_qwen = "#9B59B6"  # 파스텔 보라
    c_llama = "#5A9BD5" # 파스텔 파랑
    c_bf16 = "#B0B0B0"  # 회색
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # ==========================================
    # [Row 1] Macro View: X축을 ±7000으로 강제 동기화 (발산 증명)
    # ==========================================
    macro_xlim = (-7500, 7500)
    
    # 1-1. Qwen Macro
    axes[0, 0].hist(qwen_bf16, bins=150, range=macro_xlim, log=True, alpha=0.5, color=c_bf16, label="BF16")
    axes[0, 0].hist(qwen_2bit, bins=150, range=macro_xlim, log=True, alpha=0.8, color=c_qwen, label="2-bit (Overflow)")
    
    # 1-2. Llama Macro
    axes[0, 1].hist(llama_bf16, bins=150, range=macro_xlim, log=True, alpha=0.5, color=c_bf16, label="BF16")
    axes[0, 1].hist(llama_2bit, bins=150, range=macro_xlim, log=True, alpha=0.8, color=c_llama, label="2-bit (Contained)")
    
    for ax in axes[0, :]:
        ax.set_xlim(macro_xlim)
        ax.set_ylabel("Frequency (Log Scale)", fontweight='bold')
        ax.legend(frameon=False)
        
    # ==========================================
    # [Row 2] Micro View: X축을 ±1.5로 강제 동기화 (파편화 증명)
    # ==========================================
    micro_xlim = (-1.5, 1.5)
    
    # 2-1. Qwen Micro (Density)
    axes[1, 0].hist(qwen_bf16, bins=150, range=micro_xlim, density=True, alpha=0.5, color=c_bf16)
    axes[1, 0].hist(qwen_2bit, bins=150, range=micro_xlim, density=True, alpha=0.8, color=c_qwen)
    
    # 2-2. Llama Micro (Density)
    axes[1, 1].hist(llama_bf16, bins=150, range=micro_xlim, density=True, alpha=0.5, color=c_bf16)
    axes[1, 1].hist(llama_2bit, bins=150, range=micro_xlim, density=True, alpha=0.8, color=c_llama)
    
    for ax in axes[1, :]:
        ax.set_xlim(micro_xlim)
        ax.set_ylabel("Density (Shape Comparison)", fontweight='bold')
        ax.set_xlabel("Activation Value", fontweight='bold')

    # ==========================================
    # 글로벌 스타일링 (그리드 제거 및 회색 축)
    # ==========================================
    for ax in axes.flat:
        ax.grid(False)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(1.2)
        ax.spines['bottom'].set_linewidth(1.2)
        ax.spines['left'].set_color('gray')
        ax.spines['bottom'].set_color('gray')
        ax.tick_params(colors='gray', which='both')
        
    plt.suptitle("Terminal State Contrast: Arithmetic Overflow vs. Structural Drift", fontsize=18, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, "Fig5_P4_Contrast_Matrix.png"), dpi=300, bbox_inches='tight')
    plt.close()

# 실행 엔진에 추가
if __name__ == "__main__":
    plot_terminal_state_contrast_matrix()